# Training Pipeline — Hybrid Classical-Quantum Transfer Learning

A **dressed quantum circuit** replacing a pretrained ResNet18's final layer, ported from
[`XanaduAI/quantum-transfer-learning`](https://github.com/XanaduAI/quantum-transfer-learning/blob/master/c2q_transfer_learning_ants_bees.ipynb)
(Mari, Bromley, Izaac, Schuld & Killoran, *Transfer learning in hybrid classical-quantum neural
networks*, arXiv:1912.08278, 2019). Like [`qcnn.ipynb`](../quantum/qcnn.ipynb) and
[`quonv.ipynb`](../quantum/quonv.ipynb), there is no PennyLane dependency here — the load-bearing
circuit is ported as plain `torch` tensor ops.

Trained and evaluated on **BreastMNIST**, matching `qcnn.ipynb`, `quonv.ipynb` and the classical
notebooks in this folder — same dataset, same
[`_handlers/evaluation.py::evaluate_all_metrics`](../../../_handlers/evaluation.py) metric suite,
so all the quantum and classical runs are directly comparable. The original notebook instead
trains on the torchvision "hymenoptera_data" (ants vs. bees) set; BreastMNIST is also binary, so
the classifier head shape is unaffected — only the data source and its 3-channel adaptation change.

**The idea (classical-to-quantum, "c2q", transfer learning):** take `resnet18(pretrained=True)`,
freeze every parameter, and replace its final `fc` layer with a small hybrid module — a classical
linear layer down to 4 features, a 4-qubit variational quantum circuit, and a classical linear
layer back up to the number of classes. Only that replacement is trained.

**What's ported, in [`cnn/models/qtransfer.py`](../../models/qtransfer.py):**
- `H_layer` / `RY_layer` / `entangling_layer` — the three circuit building blocks. Notably
  `entangling_layer` is a **brick-wall** pattern (`CNOT(0,1), CNOT(2,3)` then `CNOT(1,2)` for 4
  qubits), not a ring — no wraparound `CNOT(3,0)`.
- `q_net` — Hadamard-on-every-wire (uniform superposition), `RY(q_in)` data embedding, `q_depth=6`
  repetitions of {entangling_layer; RY(q_weights[k+1])}, then per-wire `PauliZ` expectation.
  Reproduces the original's off-by-one exactly: the weight tensor is shaped `(max_layers=15,
  n_qubits)` but only rows `1..6` are ever applied — row 0 is allocated (and consumes from the
  same `torch.randn` draw) but never used by the circuit.
- `Quantumnet` (named `Quantumnet` in the original, not `DressedQuantumNet`) — the "dressed"
  wrapper: `Linear(512, 4)` pre-net, `tanh(.) * pi/2` squashing into the embedding angles, the
  quantum circuit, `Linear(4, 2)` post-net producing the class logits.

**What's replaced:** the original calls its qnode once per sample in a Python loop
(`for elem in q_in: ... torch.cat(...)`) — exactly the pattern `qcnn.py` replaces for `takh04/QCNN`.
Since the embedding angles are data-dependent per sample (unlike QCNN's fixed circuit), this port
batches only what's shareable: the post-embedding variational unitary is built once per forward
call and applied to the whole batch, while the `RY` embedding itself is a batched per-wire
rotation (no full `2^n x 2^n` unitary is built per sample).

No GPU is required for the quantum half — the whole circuit is a handful of `16x16` complex
operations per batch. A GPU does help the frozen ResNet18 feature extraction go faster.

## Install Requirements

What the pipeline imports: `torch`/`torchvision` for the model/data/training loop, `medmnist` for
BreastMNIST, `scikit-learn` for the metrics, plus `tqdm`. No `pennylane` — the circuit is
reimplemented as dense `torch` ops, so the original's quantum-simulation dependency is never
installed.

In [ ]:
!nvidia-smi

In [ ]:
!pip install torch==2.5.0 torchvision==0.20.0 torchaudio==2.5.0 --index-url https://download.pytorch.org/whl/cu124

In [ ]:
!pip install medmnist==3.0.2 scikit-learn tqdm requests

## Get the survey code

The model and handlers live in this repository's `survey/src/cnn` and `survey/src/_handlers`
packages, so the notebook needs a checkout of it. On Colab it clones into
`/content/quantum-quantization`, or `git pull --ff-only`s that directory if it is already there —
so re-running the cell after a push picks up the new code. Run locally, the notebook already sits
inside the repo, so `find_src` climbs to `survey/src` and git is never touched (your working tree
is left alone).

The branch is chosen by *environment*, not by working directory: the imports cell `os.chdir`s into
the checkout, so a cwd-based test would find `cnn` on every re-run and silently skip the pull.

If the repository is private the anonymous clone fails with an authentication error; use a token
URL instead — `REPO_URL = 'https://<GITHUB_TOKEN>@github.com/alexandrachirita98/quantum-quantization.git'`.

In [ ]:
import pathlib
import subprocess
import sys

REPO_URL = 'https://github.com/alexandrachirita98/quantum-quantization.git'

# NB: key off the environment, not the working directory. `os.chdir` in the imports cell moves the
# cwd *inside* the checkout, so a cwd-based test would report "already have the code" on every
# re-run and silently skip the pull.
IN_COLAB = 'google.colab' in sys.modules or pathlib.Path('/content').is_dir()
CLONE_DIR = pathlib.Path('/content/quantum-quantization')   # where the Colab checkout goes


def find_src(start):
    """Climb from `start` looking for the survey `src/` root — the directory holding `cnn`."""
    for p in [pathlib.Path(start), *pathlib.Path(start).parents]:
        if (p / 'cnn' / 'models' / 'qtransfer.py').is_file():
            return p
    return None


if IN_COLAB:
    if CLONE_DIR.exists():                                  # refresh whatever was cloned earlier
        subprocess.run(['git', 'pull', '--ff-only'], cwd=str(CLONE_DIR), check=True)
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(CLONE_DIR)], check=True)
    SRC = CLONE_DIR / 'survey' / 'src'
else:                                                       # local: the notebook lives in the repo
    SRC = find_src(pathlib.Path.cwd())

assert SRC is not None and (SRC / 'cnn' / 'models' / 'qtransfer.py').is_file(), f'qtransfer.py not found under {SRC}'
print('survey src:', SRC)

## Imports

`HybridResNet` and the training/data helpers come from the survey's `cnn` package;
`evaluate_all_metrics` (the same all-metrics evaluator every notebook in this folder uses) comes
from `_handlers`. `SRC` from the previous cell goes on `sys.path`, and `os.chdir` moves into it so
`./data` (the shared [`src/data`](../../../data) folder) is where `medmnist` downloads BreastMNIST.

In [ ]:
import os
import sys

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
from torch.optim import lr_scheduler

# import the pipeline from the survey `src/` root, and work from there so './data' resolves inside it
sys.path.insert(0, str(SRC))
os.chdir(SRC)
print('working directory:', os.getcwd())

# drop cached modules, so a `git pull` above is actually reflected on a re-run
for _m in [m for m in list(sys.modules) if m in ('cnn', '_handlers') or m.startswith(('cnn.', '_handlers.'))]:
    del sys.modules[_m]

from cnn.handlers.qtransfer import (
    build_breastmnist_datasets,
    HybridResNet,
    train_hybrid,
    HybridLogits,
)
from _handlers.evaluation import evaluate_all_metrics

## Configuration

The notebook's own hyperparameter cell, kept as a plain namespace instead of loose globals:
4 qubits, `q_depth=6` variational layers (`max_layers=15` kept even though only 6 are used,
matching the original's own comment), `q_delta=0.01` initial weight spread, Adam at `lr=0.0004`
with `StepLR` decaying by `gamma=0.1` every 10 epochs, batch size 4.

**Dataset** — `breastmnist`, the same medmnist flag
[`resnet50.ipynb`](../../notebooks/classical/resnet50.ipynb) and `qcnn.ipynb` train on, at its
native 28x28 resolution (upsampled to ResNet18's expected 224x224 below).

In [ ]:
from types import SimpleNamespace

args = SimpleNamespace(
    dataset='breastmnist',                            # a medmnist flag, native 28x28 resolution
    n_qubits=4,
    q_depth=6,
    max_layers=15,                                    # "Keep 15 even if not all are used."
    q_delta=0.01,                                     # initial spread of random quantum weights
    lr=0.0004,
    batch_size=4,
    num_epochs=3,                                     # paper default is 30; kept small for a demo run
    step_size=10,                                     # StepLR: decay every 10 epochs
    gamma=0.1,                                        # StepLR: decay factor
    rng_seed=0,
)
torch.manual_seed(args.rng_seed)

## Device

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using {} device.".format(device))

## Dataset

BreastMNIST (malignant vs. normal/benign, already binary). `build_breastmnist_datasets` downloads
the native 28x28 grayscale `.npz`, replicates the single channel to 3 and applies the original's
`data_transforms` (`Resize(256)`, `CenterCrop(224)`, ImageNet normalization) — required because
`resnet18(pretrained=True)` expects 3-channel ImageNet-statistics input regardless of the
downstream task. `'train'`/`'val'` are medmnist's own splits.

In [ ]:
image_datasets = build_breastmnist_datasets(root='./data')

dataloaders = {
    phase: data.DataLoader(image_datasets[phase], batch_size=args.batch_size, shuffle=True)
    for phase in ('train', 'val')
}
dataset_sizes = {phase: len(image_datasets[phase]) for phase in ('train', 'val')}
nb_classes = 2

print('dataset sizes:', dataset_sizes)

## Model

`HybridResNet` wraps a frozen `resnet18(pretrained=True)` with its `.fc` replaced by `Quantumnet` —
the dressed quantum circuit described above. Only `model.fc`'s parameters (`pre_net`, `q_params`,
`post_net`) are ever trained; the backbone stays a fixed ImageNet feature extractor.

In [ ]:
model = HybridResNet(n_qubits=args.n_qubits, q_depth=args.q_depth, max_layers=args.max_layers,
                     q_delta=args.q_delta, out_features=nb_classes).to(device)

print(model.fc)
print('trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

## Optimizer, Scheduler & Loss

Adam over `model.fc.parameters()` only (matching the notebook's
`optim.Adam(model_hybrid.fc.parameters(), lr=step)`), `StepLR` decaying the learning rate by
`gamma` every `step_size` epochs, `nn.CrossEntropyLoss` for the class logits.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=args.lr)
scheduler = lr_scheduler.StepLR(optimizer, step_size=args.step_size, gamma=args.gamma)

## Train

`train_hybrid` (from the handler) runs `args.num_epochs` train/val passes, restoring the
best-val-accuracy weights at the end, matching the original's `train_model`.

In [ ]:
model = train_hybrid(model, criterion, optimizer, scheduler, dataloaders, dataset_sizes, device,
                     num_epochs=args.num_epochs)

## Evaluate all metrics

Same helper every notebook in this folder uses:
[`evaluate_all_metrics`](../../../_handlers/evaluation.py) runs the model once over one split and
prints **every** metric the training routines can produce — the medmnist Evaluator AUC/ACC plus
accuracy, weighted precision / recall (sensitivity) / F1, per-class + average specificity,
one-vs-rest AUC, the confusion matrix and a per-class report.

`HybridLogits` is a thin pass-through (the model already returns raw logits). `size=28` tells the
medmnist `Evaluator` to score against the native-resolution `.npz` (the model only ever sees the
upsampled 224x224 RGB copy). Set `split` to `'train'` or `'val'`.

In [ ]:
split = 'val'   # 'train' or 'val'

eval_net = HybridLogits(model)
metrics = evaluate_all_metrics(eval_net, image_datasets[split], args.dataset, nb_classes=nb_classes,
                               device=device, split=split, batch_size=2 * args.batch_size, size=28)